# Retrieval Augmented Generation (RAG)
 
<img src="https://weaviate.io/assets/images/rag-b22b743c4f915c854a1501e44c3f9784.png" width="50%">

In [ ]:
# source qdrant_env/bin/activate
##“Na primeira execução, o modelo de embeddings é baixado localmente. Isso pode levar alguns minutos e não mostra progresso.”
# -------------------------------
# QDRANT
# -------------------------------
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
# -------------------------------
# LANGCHAIN (OPEN-SOURCE)
# -------------------------------
from langchain_qdrant import QdrantVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch
from transformers import pipeline
from langchain.llms import HuggingFacePipeline  


In [ ]:
# !pip install transformers accelerate sentencepiece
# !pip install langchain-community langchain-qdrant

In [ ]:
# -------------------------------
# QDRANT SETUP
# -------------------------------
client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "RAG-DB"

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=384,                    # MiniLM
        distance=Distance.COSINE
    ),
)

# -------------------------------
# EMBEDDING - OPEN SOURCE
# -------------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


def get_chunks(text):
    splitter = RecursiveCharacterTextSplitter( #preserva melhor o contexto que CharacterTextSplitter
        chunk_size=300, #150
        chunk_overlap=50, #20
        separators=["\n\n", "\n", " ", ""]
    )
    return splitter.split_text(text)

# -------------------------------
# LOAD DATA
# -------------------------------
with open("base_dados.txt", encoding="utf-8") as f:
    raw_text = f.read()

texts = get_chunks(raw_text)
vectorstore.add_texts(texts)



['afe659794f2641e19012316f7a00eb08',
 '706d9cec4c994e1199af94b8431a3e6c',
 'ec6467c111d24e3b98ddb46754065853',
 '0569329827fd4e249e58bf615bf49947',
 'c84a3f1598c04e9da67dd662ddc612dd',
 '78dd911995db436ba66b0072d2388696',
 '4ba4cce135c341b395fc3b9a7d62a9b0',
 '2b50bb59b65b4797a13b80bf316ab656',
 '299eea9607c34550ab2a4cdcee5ecb47',
 'a15cbc32ae434317940aa7f4d5915089',
 'f027897b0a1d49eda139300878cd93da',
 '3419bb01510543bb8ca081a54d4e8ebe',
 '10ca82ce0a2a4b9180794bca69892343',
 'bda933667f744aea8e433848d18553b8',
 '2867510b29a645b1b82b011d1c3576c8',
 '48c81655e18c4353b30e612636f0931e',
 '869dd43177024f6281b2e4f8b9f558fd',
 '74d417bd056840d389743eb76f470b8f',
 'a53334fd0a5d405b937058bd20fb8199',
 'ff80f9d36dbb4b638fcf40b72f6cc9d2',
 'b044db41345a41b783adc7ae1aeb89c4',
 'dc16a97ffc6a41ef90764a06267e0f65',
 '94fba6a5cb8f4484a53c34d76ac2563c',
 '4d382bbc2de647dcb0ce7d34a38bbe49',
 'dbff3fcc20d346f3994825275a913972',
 'a546ae2eae54436599d36dd53603551b',
 '7872e3570148492497714bdda1d4a90c',
 

### LLM

In [4]:
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base", #"google/flan-t5-small",
    
    #"text-generation",
    #model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", #muito pesado
    dtype=torch.float32, #desativar se tiver gpu
    device="cpu", # device="cpu", device="mps"
    max_new_tokens=64 #64
)

llm = HuggingFacePipeline(pipeline=pipe)



Device set to use cpu
/var/folders/__/rsjg1yys7y904vmmv_qqyptr0000gn/T/ipykernel_46258/1904477407.py:12: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [10]:
#-------------------------------
# RAG
# -------------------------------
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
def perguntar_rag(query):

    docs = retriever.invoke(query)
    context = ""
    for i, doc in enumerate(docs, start=1):
        # print(f"[Chunk {i}]")
        # print(doc.page_content)
        # print("\n------------------\n")
        context += doc.page_content + "\n"

    # prompt = f"""Contexto:{context} Pergunta:{query} """
    prompt = f"""Use o contexto abaixo para responder à pergunta de forma curta e direta.
        Contexto: {context}
        Pergunta: {query}
        Resposta:"""

    print("\n=== PROMPT ===\n")
    print(prompt)

    response = pipe(prompt)
    print("\n=== RAW RESPONSE ===\n")
    print(response)

    print("\n=== RESPOSTA FINAL ===\n")
    print(response[0]["generated_text"])

### Query 1

In [11]:

# -------------------------------
# QUERIES
# -------------------------------
query = "quem avalia a solicitação de compra?"

perguntar_rag(query)



=== PROMPT ===

Use o contexto abaixo para responder à pergunta de forma curta e direta.
        Contexto: Após o preenchimento, a solicitação é encaminhada ao departamento de Compras para avaliação.
Passo 2: Análise da Solicitação
Passo 1: Solicitação de Compra
O departamento de Compras é responsável por receber e analisar as solicitações de compra.

        Pergunta: quem avalia a solicitação de compra?
        Resposta:

=== RAW RESPONSE ===

[{'generated_text': 'Após o preenchimento, a solicitaço é encaminhada ao departamento de Compras para avaliaço. Passo 2: Análise da Solicitaço Passo'}]

=== RESPOSTA FINAL ===

Após o preenchimento, a solicitaço é encaminhada ao departamento de Compras para avaliaço. Passo 2: Análise da Solicitaço Passo


### Query 2

In [12]:
query = "quantos por cento das solicitações são revisadas pelo comitê?"
perguntar_rag(query)
#Resposta: 10% das solicitações são revisadas pelo comitê de revisão composto por membros de diferentes departamentos.


=== PROMPT ===

Use o contexto abaixo para responder à pergunta de forma curta e direta.
        Contexto: A solicitação deve conter detalhes precisos, como descrição do item, quantidade necessária, justificativa, e qualquer informação adicional relevante.
a solicitação à aprovação final.
Após o preenchimento, a solicitação é encaminhada ao departamento de Compras para avaliação.
Passo 2: Análise da Solicitação

        Pergunta: quantos por cento das solicitações são revisadas pelo comitê?
        Resposta:

=== RAW RESPONSE ===

[{'generated_text': 'A solicitaço debe contar detalhes precisos, como descriço do item, quantidade necessária, justificativa, e qualquer informaço adicional relevante.'}]

=== RESPOSTA FINAL ===

A solicitaço debe contar detalhes precisos, como descriço do item, quantidade necessária, justificativa, e qualquer informaço adicional relevante.


#### Query 3

In [13]:
query ="Qual a função do setor financeiro no processo de compra?"
perguntar_rag(query)


=== PROMPT ===

Use o contexto abaixo para responder à pergunta de forma curta e direta.
        Contexto: O setor financeiro é responsável por validar a disponibilidade orçamentária antes da emissão da ordem de compra.
A ordem de compra é emitida apenas após a confirmação da disponibilidade orçamentária pelo setor financeiro.
A ordem de compra é encaminhada ao fornecedor, com cópia para o solicitante e para o setor financeiro para registro e controle.

        Pergunta: Qual a função do setor financeiro no processo de compra?
        Resposta:

=== RAW RESPONSE ===

[{'generated_text': 'O setor financeiro é responsável para validar a disponibilidade orçamentária antes da emisso da ordem de compra. A ordem de compra é emitida apenas '}]

=== RESPOSTA FINAL ===

O setor financeiro é responsável para validar a disponibilidade orçamentária antes da emisso da ordem de compra. A ordem de compra é emitida apenas 
